# Construction de l'index FAISS

**Objectif :** embedder chaque corpus avec `mistral-embed` et construire un index FAISS sérialisé dans `data/faiss_index/`.

Étapes :
1. Chargement des données traitées
2. Initialisation du modèle d'embedding Mistral
3. Génération des embeddings (par batch — rate limiting Mistral)
4. Construction et sauvegarde de l'index FAISS

In [1]:
import os
import time
import pandas as pd
import faiss
import numpy as np
from langchain_mistralai import MistralAIEmbeddings

df = pd.read_csv("../data/processed/events_clean.csv")
print(f"{len(df)} événements chargés")

4966 événements chargés


## 2. Initialisation du modèle d'embedding

`mistral-embed` produit des vecteurs de dimension 1024. On teste sur un exemple avant de lancer la totalité.

In [2]:
embeddings = MistralAIEmbeddings(
    model="mistral-embed",
    mistral_api_key=os.getenv("MISTRAL_API_KEY")
)

# Test sur un seul texte
test_vector = embeddings.embed_query(df["corpus"].iloc[0])
print(f"Dimension du vecteur : {len(test_vector)}")

/home/zile/workspace/rennes-agenda-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dimension du vecteur : 1024


## 3. Génération des embeddings

On envoie les textes par batches de 50 avec une pause d'1 seconde entre chaque batch pour respecter le rate limiting de l'API Mistral.

In [ ]:
corpus_list = df["corpus"].tolist()
all_vectors = []
batch_size = 50  # limite conservative pour éviter les erreurs 429

for i in range(0, len(corpus_list), batch_size):
    batch = corpus_list[i:i + batch_size]
    vectors = embeddings.embed_documents(batch)
    all_vectors.extend(vectors)
    print(f"{len(all_vectors)}/{len(corpus_list)}", end="\r")
    time.sleep(1)  # pause entre chaque batch

print(f"\nEmbeddings générés : {len(all_vectors)}")

## 4. Construction et sauvegarde de l'index FAISS

On crée un index `IndexFlatL2` — recherche exacte par distance euclidienne, adapté à notre volume (< 10 000 documents).  
FAISS ne persiste pas automatiquement : on doit sérialiser explicitement avec `faiss.write_index`.

In [ ]:
# FAISS attend un tableau numpy float32
vectors_np = np.array(all_vectors, dtype=np.float32)
dimension = vectors_np.shape[1]

# IndexFlatL2 : recherche exacte par distance euclidienne
index = faiss.IndexFlatL2(dimension)
index.add(vectors_np)
print(f"Index FAISS : {index.ntotal} vecteurs de dimension {dimension}")

os.makedirs("../data/faiss_index", exist_ok=True)
faiss.write_index(index, "../data/faiss_index/events.index")
# Mapping position → uid/corpus, nécessaire pour retrouver le texte après recherche
df[["uid", "corpus"]].to_csv("../data/faiss_index/events_mapping.csv", index=True)
print("Index et mapping sauvegardés dans data/faiss_index/")